# 2. Interior series and Theorem 2 (Sections 3–4)

Checks, against the reference of notebook 1:

* the polynomial form (Eqs. *poly*, *A*) equals the partial sum $S_M$ (Eq. *SM*), and Eq. *S3*;
* **Theorem 2**: $|S_M-D_\gamma|\le B_M$, the sign of $D_\gamma-S_M$ is $(-1)^{\lceil\gamma\rceil}$,
  and $B_M$ increases with $z$;
* **Corollary 1** (the cut $z_c$): pizzetti's interior series has flux error $\le$ `series_tol`
  for $z\le z_c$, judged by exoplanet-core (Agol et al. 2020) on dense grids.

The paper's formulas are transcribed into `refcheck.py` from the text (not from pizzetti).
Near $z=0$ the true errors are tiny (down to $10^{-40}$), so Theorem 2 uses the high-precision
reference and only judges points where the error or the bound exceeds $10^{-26}$.

In [1]:
import time
import numpy as np
import mpmath as mp
import refcheck as rc

FULL = False          # True: the larger grids (minutes to hours)
t_start = time.time()

mp.mp.dps = 40

## 2a. Polynomial form and Eq. (S3)

In [2]:
worst = 0
for g in [mp.mpf("0.3"), mp.mpf("0.5"), mp.mpf("1.7")]:
    for M in [3, 11]:
        for z, p in [(0.0, 0.1), (0.4, 0.2), (0.7, 0.25)]:
            z, p = mp.mpf(z), mp.mpf(p)
            worst = max(worst, abs(rc.S_poly(M, z, p, g) - rc.S(M, z, p, g)))
print("max |Eq.(poly) - Eq.(SM)| =", mp.nstr(worst, 3))

worst = 0
for z, p in [(0.1, 0.05), (0.5, 0.2), (0.0, 0.3)]:
    z, p = mp.mpf(z), mp.mpf(p)
    v = 1 / (1 - z * z)
    s3 = mp.sqrt(1 - z * z) * (1 - p**2 / 8 * (v + v * v) + p**4 / 12 * v * v - p**4 / 8 * v**3 - p**6 / 64 * v**3)
    worst = max(worst, abs(s3 - rc.S(3, z, p, mp.mpf(0.5))))
print("max |Eq.(S3) - S_3| =", mp.nstr(worst, 3))

max |Eq.(poly) - Eq.(SM)| = 2.3e-41
max |Eq.(S3) - S_3| = 1.15e-41


## 2b. Theorem 2

In [3]:
FLOOR = mp.mpf("1e-26")
gammas = ["0.25", "0.3", "0.5", "0.75", "1.25", "1.7"] if FULL else ["0.3", "0.5", "1.7"]
ps = [0.0092, 0.05, 0.1, 0.2, 0.3] if FULL else [0.0092, 0.1, 0.3]
nz = 15 if FULL else 6
Ms = [1, 3, 5, 11, 21] if FULL else [3, 11]
judged = viol = sign_err = nonmono = 0
ratios = []
for gs in gammas:
    g = mp.mpf(gs)
    for p in ps:
        pm = mp.mpf(p)
        zs = [mp.mpf(x) for x in np.linspace(0, 0.999 * (1 - p), nz)]
        Ds = [rc.D_mp(z, pm, g, dps=20) for z in zs]
        for M in Ms:
            Bs = [rc.B(M, z, pm, g) for z in zs]
            nonmono += any(Bs[i + 1] < Bs[i] for i in range(len(Bs) - 1))
            for z, d, b in zip(zs, Ds, Bs):
                err = d - rc.S(M, z, pm, g)
                if abs(err) < FLOOR and b < FLOOR:
                    continue
                judged += 1
                ratios.append(float(abs(err) / b))
                viol += abs(err) > b
                sign_err += abs(err) > FLOOR and mp.sign(err) != (-1) ** int(mp.ceil(g))
print(f"points judged: {judged}; bound violated: {viol}; wrong sign: {sign_err}; B_M not increasing: {nonmono}")
print(f"error / bound: max {max(ratios):.5f}, median {np.median(ratios):.3f}")
assert viol == 0 and sign_err == 0 and nonmono == 0

points judged: 95; bound violated: 0; wrong sign: 0; B_M not increasing: 0
error / bound: max 0.99997, median 0.927


## 2c. Corollary 1: the cut, judged by exoplanet-core (Agol et al. 2020)

Quadratic law, pizzetti's interior series (Eq. *final*) on dense grids $0\le z\le z_c$.

In [4]:
try:
    import exoplanet_core as ec
    from pizzetti import occult, series_cut
except ImportError as e:
    ec = None
    print("skipped:", e)

if ec is not None:
    n = 0
    worst = 0.0
    ps = [0.001, 0.0092, 0.05, 0.1, 0.2, 0.3, 0.4] if FULL else [0.0092, 0.1, 0.3]
    for p in ps:
        for tau in [1e-8, 1e-9, 1e-10, 1e-12]:
            for u in [(0.4, 0.25), (1.0, 0.0), (0.1, 0.8)]:
                zc = series_cut(p, "quadratic", list(u), tau)
                z = np.linspace(0, zc, 20001 if FULL else 2001)
                z = z[np.abs(z - p) > 1e-12]
                fp = occult(z, p, list(u), "quadratic", series_tol=tau)
                fa = ec.quad_limbdark_light_curve(u[0], u[1], z, np.full(z.size, p)) + 1
                e = np.max(np.abs(fp - fa))
                worst = max(worst, e / tau)
                n += 1
                assert e <= tau + 2e-15, (p, tau, u, e)
    print(f"{n} light-curve segments; largest error / series_tol = {worst:.3f}")

36 light-curve segments; largest error / series_tol = 0.938


In [5]:
print(f"notebook run time: {time.time() - t_start:.0f} s")

notebook run time: 56 s
